# Create one PEXL project from multiple timestep export files

Use this workflow when the scenarios to compare are stored in **separate PEExcel timestep export files**.

Each input workbook is loaded as a scenario with its scalar `IN` / `OUT` values and its hourly timestep data. `Project.from_files()` combines the individual scenarios into one `Project`.

This notebook only covers:

1. selecting the timestep export files,
2. combining them into one PEXL project,
3. checking that all scenarios contain timestep data,
4. saving the assembled project for later analysis.

The saved project is used by `howto_analyze_scenarios.ipynb`.


## 1. Set input files and output project

Edit the input folder and file pattern so that they match the timestep exports you want to combine.

The assembled project is saved as a Python pickle (`.pkl`). This preserves the complete in-memory PEXL project, including the attached hourly timestep data.

Only load pickle files that you created yourself.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pickle

import pandas as pd
import pexl


# Folder containing the individual PEExcel timestep exports.
input_folder = Path(r"C:\Users\schneids\OneDrive\Documents")
file_pattern = "Aspern PVCheck*.xlsx"

# Saved multi-scenario project used by the analysis notebook.
project_path = Path(
    r"C:\Users\schneids\Nextcloud\EE\6_Daten\Quartiere"
    r"\1220_Wien_Aspern_Seestadt\Unterlagen PV Überschüsse"
    r"\PEexcel Auswertung\aspern_pvcheck_project.pkl"
)


## 2. Find the timestep export files

Temporary Excel files beginning with `~$` are ignored. Review the resulting list before creating the project.


In [2]:
files = [
    file
    for file in sorted(input_folder.glob(file_pattern))
    if not file.name.startswith("~$")
]

if not files:
    raise FileNotFoundError(
        f"No files matching {file_pattern!r} found in {input_folder}"
    )

pd.DataFrame(
    {
        "file": [file.name for file in files],
        "path": [str(file) for file in files],
    }
)


,file,path
0,Aspern PVCheck _ E7A.xlsx,C:\Users\schneids\OneDrive\Documents\Aspern PV...
1,Aspern PVCheck _ F10.xlsx,C:\Users\schneids\OneDrive\Documents\Aspern PV...
2,Aspern PVCheck _ F12.xlsx,C:\Users\schneids\OneDrive\Documents\Aspern PV...
3,Aspern PVCheck _ F13.xlsx,C:\Users\schneids\OneDrive\Documents\Aspern PV...
4,Aspern PVCheck _ F3.xlsx,C:\Users\schneids\OneDrive\Documents\Aspern PV...
5,Aspern PVCheck _ F5.xlsx,C:\Users\schneids\OneDrive\Documents\Aspern PV...
6,Aspern PVCheck _ F6.xlsx,C:\Users\schneids\OneDrive\Documents\Aspern PV...
7,Aspern PVCheck _ F7.xlsx,C:\Users\schneids\OneDrive\Documents\Aspern PV...
8,Aspern PVCheck _ F9B.xlsx,C:\Users\schneids\OneDrive\Documents\Aspern PV...
9,Aspern PVCheck _ H1.xlsx,C:\Users\schneids\OneDrive\Documents\Aspern PV...


## 3. Combine all files into one project

`Project.from_files()` reads the individual exports and combines their scenarios into one ordered `Project`.

The scenario identity from each export is retained, and each scenario carries its own timestep dataset.


In [3]:
project = pexl.Project.from_files(files)
project


<Project scenarios=11 warnings=0>

## 4. Check the imported scenarios

Before saving, verify that every scenario has timestep data. The summary also shows the scenario column name and the number of imported hourly rows and variables.


In [4]:
missing_timeseries = [
    scenario.column_name
    for scenario in project
    if scenario.timeseries is None
]

if missing_timeseries:
    raise ValueError(
        f"Missing timestep data for scenarios: {missing_timeseries}"
    )

scenario_summary = pd.DataFrame(
    [
        {
            "column_name": scenario.column_name,
            "project_name": scenario.project_name,
            "scenario_name": scenario.name,
            "timesteps": len(scenario.timeseries),
            "timeseries_variables": len(scenario.timeseries.columns),
        }
        for scenario in project
    ]
)

scenario_summary


,column_name,project_name,scenario_name,timesteps,timeseries_variables
0,Aspern PVCheck _ E7A,Aspern PVCheck,E7A,8760,15
1,Aspern PVCheck _ F10,Aspern PVCheck,F10,8760,15
2,Aspern PVCheck _ F12,Aspern PVCheck,F12,8760,15
3,Aspern PVCheck _ F13,Aspern PVCheck,F13,8760,15
4,Aspern PVCheck _ F3,Aspern PVCheck,F3,8760,15
5,Aspern PVCheck _ F5,Aspern PVCheck,F5,8760,15
6,Aspern PVCheck _ F6,Aspern PVCheck,F6,8760,15
7,Aspern PVCheck _ F7,Aspern PVCheck,F7,8760,15
8,Aspern PVCheck _ F9B,Aspern PVCheck,F9B,8760,15
9,Aspern PVCheck _ H1,Aspern PVCheck,H1,8760,15


## 5. Save the assembled project

The project is serialized once so that later analysis notebooks do not need to reopen and combine all source workbooks.

If the PEXL object model or generated schema changes substantially, recreate the pickle from the original Excel exports rather than treating it as a permanent interchange format.


In [5]:
project_path.parent.mkdir(parents=True, exist_ok=True)

with project_path.open("wb") as file:
    pickle.dump(
        project,
        file,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

print(f"Saved project to: {project_path}")


Saved project to: C:\Users\schneids\Nextcloud\EE\6_Daten\Quartiere\1220_Wien_Aspern_Seestadt\Unterlagen PV Überschüsse\PEexcel Auswertung\aspern_pvcheck_project.pkl


## 6. Optional verification

Reload the saved project once and verify that the same scenarios are available. The analysis notebook will start from this saved file.


In [ ]:
with project_path.open("rb") as file:
    saved_project = pickle.load(file)

[
    scenario.column_name
    for scenario in saved_project
] == [
    scenario.column_name
    for scenario in project
]


True

In [9]:

saved_project

<Project scenarios=11 warnings=0>

The multi-scenario project is now ready for `howto_analyze_scenarios.ipynb`.
